# LVLM Data Exploration

This notebook explores the TVQA and ActivityNet-QA datasets used for training and evaluating the Large Language Video Model (LVLM).

**Key Analyses:**
- Dataset statistics and distributions
- QA pair analysis
- Temporal span distributions
- Video length statistics
- Question complexity analysis

In [ ]:
# Import libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Set paths
data_dir = Path('../data/datasets')
cache_dir = Path('../data/cache')

print(f"Data directory: {data_dir}")
print(f"Cache directory: {cache_dir}")

## 1. Load TVQA Dataset

In [ ]:
# Try to load TVQA annotations
tvqa_files = list((data_dir / 'tvqa').glob('*.json')) if (data_dir / 'tvqa').exists() else []

if tvqa_files:
    # Load training split
    train_file = [f for f in tvqa_files if 'train' in f.stem]
    if train_file:
        with open(train_file[0]) as f:
            tvqa_data = json.load(f)
        print(f"Loaded TVQA dataset: {len(tvqa_data)} QA pairs")
        print(f"From file: {train_file[0].name}")
    else:
        print("No TVQA train file found")
        tvqa_data = None
else:
    print("TVQA dataset not found. Download from: https://tvqa.cs.utexas.edu/")
    tvqa_data = None

In [ ]:
if tvqa_data:
    # Analyze TVQA structure
    sample = tvqa_data[0]
    print("Sample TVQA QA pair:")
    print(json.dumps(sample, indent=2)[:500])
    print("\n...")

## 2. TVQA Statistics

In [ ]:
if tvqa_data:
    # Extract statistics
    video_ids = [qa.get('video_id') for qa in tvqa_data if 'video_id' in qa]
    unique_videos = len(set(video_ids))
    
    # Question lengths
    q_lengths = [len(qa.get('question', '').split()) for qa in tvqa_data if 'question' in qa]
    
    # Temporal spans
    spans = [qa.get('ts', [0, 1]) for qa in tvqa_data if 'ts' in qa]
    span_durations = [s[1] - s[0] for s in spans]
    
    print("TVQA Dataset Statistics:")
    print(f"  Total QA pairs: {len(tvqa_data)}")
    print(f"  Unique videos: {unique_videos}")
    print(f"  Pairs per video: {len(tvqa_data) / unique_videos:.1f}")
    print(f"\nQuestion Statistics:")
    print(f"  Avg words: {np.mean(q_lengths):.1f}")
    print(f"  Max words: {np.max(q_lengths)}")
    print(f"  Min words: {np.min(q_lengths)}")
    print(f"\nTemporal Span Statistics:")
    print(f"  Avg duration: {np.mean(span_durations):.2f}s")
    print(f"  Max duration: {np.max(span_durations):.2f}s")
    print(f"  Min duration: {np.min(span_durations):.2f}s")

In [ ]:
if tvqa_data:
    # Visualize distributions
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Question length distribution
    axes[0, 0].hist(q_lengths, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Question Length (words)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('TVQA Question Length Distribution')
    
    # Temporal span duration
    axes[0, 1].hist(span_durations, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Temporal Span Duration (seconds)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('TVQA Temporal Span Duration Distribution')
    
    # QA pairs per video
    qa_per_video = Counter(video_ids)
    axes[1, 0].hist(list(qa_per_video.values()), bins=20, color='mediumseagreen', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('QA Pairs per Video')
    axes[1, 0].set_ylabel('Number of Videos')
    axes[1, 0].set_title('TVQA QA Pairs Distribution per Video')
    
    # Answer type distribution (if available)
    answer_types = [qa.get('answer_type', 'unknown') for qa in tvqa_data if 'answer_type' in qa]
    if answer_types:
        answer_counts = Counter(answer_types)
        axes[1, 1].bar(answer_counts.keys(), answer_counts.values(), color='mediumpurple', edgecolor='black', alpha=0.7)
        axes[1, 1].set_xlabel('Answer Type')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('TVQA Answer Type Distribution')
        axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 3. Load ActivityNet-QA Dataset

In [ ]:
# Try to load ActivityNet-QA annotations
activitynet_files = list((data_dir / 'activitynet_qa').glob('*.json')) if (data_dir / 'activitynet_qa').exists() else []

if activitynet_files:
    # Load first available file
    with open(activitynet_files[0]) as f:
        activitynet_raw = json.load(f)
    
    # Convert to list format
    if isinstance(activitynet_raw, dict):
        activitynet_data = list(activitynet_raw.values())
    else:
        activitynet_data = activitynet_raw
    
    print(f"Loaded ActivityNet-QA dataset: {len(activitynet_data)} QA pairs")
    print(f"From file: {activitynet_files[0].name}")
else:
    print("ActivityNet-QA dataset not found. Download from: https://github.com/lixiangpeng/AGQA")
    activitynet_data = None

In [ ]:
if activitynet_data:
    # Analyze ActivityNet-QA structure
    sample = activitynet_data[0]
    print("Sample ActivityNet-QA QA pair:")
    print(json.dumps(sample, indent=2)[:500])
    print("\n...")

## 4. ActivityNet-QA Statistics

In [ ]:
if activitynet_data:
    # Extract statistics
    video_ids_an = [qa.get('video_id', qa.get('vid', '')) for qa in activitynet_data if 'video_id' in qa or 'vid' in qa]
    unique_videos_an = len(set(video_ids_an))
    
    # Question lengths
    q_lengths_an = [len(qa.get('question', '').split()) for qa in activitynet_data if 'question' in qa]
    
    # Temporal spans
    spans_an = [qa.get('ts', [0, 1]) for qa in activitynet_data if 'ts' in qa]
    span_durations_an = [s[1] - s[0] for s in spans_an]
    
    print("ActivityNet-QA Dataset Statistics:")
    print(f"  Total QA pairs: {len(activitynet_data)}")
    print(f"  Unique videos: {unique_videos_an}")
    print(f"  Pairs per video: {len(activitynet_data) / unique_videos_an:.1f}")
    print(f"\nQuestion Statistics:")
    print(f"  Avg words: {np.mean(q_lengths_an):.1f}")
    print(f"  Max words: {np.max(q_lengths_an)}")
    print(f"  Min words: {np.min(q_lengths_an)}")
    print(f"\nTemporal Span Statistics:")
    print(f"  Avg duration: {np.mean(span_durations_an):.2f}s")
    print(f"  Max duration: {np.max(span_durations_an):.2f}s")
    print(f"  Min duration: {np.min(span_durations_an):.2f}s")

In [ ]:
if activitynet_data:
    # Visualize distributions
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Question length distribution
    axes[0, 0].hist(q_lengths_an, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Question Length (words)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('ActivityNet-QA Question Length Distribution')
    
    # Temporal span duration
    axes[0, 1].hist(span_durations_an, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Temporal Span Duration (seconds)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('ActivityNet-QA Temporal Span Duration Distribution')
    
    # QA pairs per video
    qa_per_video_an = Counter(video_ids_an)
    axes[1, 0].hist(list(qa_per_video_an.values()), bins=20, color='mediumseagreen', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('QA Pairs per Video')
    axes[1, 0].set_ylabel('Number of Videos')
    axes[1, 0].set_title('ActivityNet-QA QA Pairs Distribution per Video')
    
    # Video ID distribution
    axes[1, 1].text(0.5, 0.5, f'Total QA pairs: {len(activitynet_data)}\nUnique videos: {unique_videos_an}\nAvg pairs/video: {len(activitynet_data)/unique_videos_an:.1f}',
                    ha='center', va='center', fontsize=14, transform=axes[1, 1].transAxes)
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

## 5. Dataset Comparison

In [ ]:
# Compare datasets
comparison_data = {
    'Dataset': ['TVQA', 'ActivityNet-QA'],
    'Total QA pairs': [len(tvqa_data) if tvqa_data else 0, len(activitynet_data) if activitynet_data else 0],
    'Unique videos': [unique_videos if tvqa_data else 0, unique_videos_an if activitynet_data else 0],
    'Avg question length': [np.mean(q_lengths) if tvqa_data else 0, np.mean(q_lengths_an) if activitynet_data else 0],
    'Avg span duration': [np.mean(span_durations) if tvqa_data else 0, np.mean(span_durations_an) if activitynet_data else 0],
}

comp_df = pd.DataFrame(comparison_data)
print("\nDataset Comparison:")
print(comp_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

datasets = comp_df['Dataset'].tolist()

# Total QA pairs
axes[0, 0].bar(datasets, comp_df['Total QA pairs'], color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Number of QA pairs')
axes[0, 0].set_title('Total QA Pairs Comparison')
for i, v in enumerate(comp_df['Total QA pairs']):
    axes[0, 0].text(i, v + 2000, f'{int(v):,}', ha='center', va='bottom', fontweight='bold')

# Unique videos
axes[0, 1].bar(datasets, comp_df['Unique videos'], color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[0, 1].set_ylabel('Number of videos')
axes[0, 1].set_title('Unique Videos Comparison')
for i, v in enumerate(comp_df['Unique videos']):
    axes[0, 1].text(i, v + 100, f'{int(v):,}', ha='center', va='bottom', fontweight='bold')

# Average question length
axes[1, 0].bar(datasets, comp_df['Avg question length'], color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel('Average words')
axes[1, 0].set_title('Average Question Length Comparison')
for i, v in enumerate(comp_df['Avg question length']):
    axes[1, 0].text(i, v + 0.1, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Average span duration
axes[1, 1].bar(datasets, comp_df['Avg span duration'], color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('Average seconds')
axes[1, 1].set_title('Average Temporal Span Duration Comparison')
for i, v in enumerate(comp_df['Avg span duration']):
    axes[1, 1].text(i, v + 0.5, f'{v:.1f}s', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Summary and Insights

In [ ]:
print("\n=== LVLM Dataset Analysis Summary ===")
print("\nKey Insights:")
print("\n1. Dataset Sizes:")
print(f"   - TVQA is larger (152k pairs) with more videos (21.8k)")
print(f"   - ActivityNet-QA is smaller (58k pairs) but represents longer videos")
print("\n2. Question Complexity:")
print(f"   - TVQA questions: avg {np.mean(q_lengths) if tvqa_data else 'N/A':.1f} words")
print(f"   - ActivityNet-QA questions: avg {np.mean(q_lengths_an) if activitynet_data else 'N/A':.1f} words")
print("\n3. Temporal Reasoning:")
print(f"   - TVQA spans: avg {np.mean(span_durations) if tvqa_data else 'N/A':.2f}s (short-term)")
print(f"   - ActivityNet-QA spans: avg {np.mean(span_durations_an) if activitynet_data else 'N/A':.2f}s (longer-term)")
print("\n4. Data Composition:")
print(f"   - TVQA: {len(tvqa_data) / unique_videos:.1f} if tvqa_data else 'N/A' QA pairs per video on avg")
print(f"   - ActivityNet-QA: {len(activitynet_data) / unique_videos_an:.1f} if activitynet_data else 'N/A' QA pairs per video on avg")
print("\n5. Model Training Implications:")
print("   - Large dataset (TVQA) good for initial pre-training")
print("   - Longer videos (ActivityNet-QA) test temporal binding effectiveness")
print("   - Multi-hop reasoning needed for temporal spans")
print("   - Adaptive depth important for varying complexity")